## Setup

In [ ]:
import sys
sys.path.append('src')

from sentence_transformers import SentenceTransformer
from neo4j import GraphDatabase
from vector_store import FaissStore
from config import NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, EMBEDDING_MODEL
import pandas as pd
import json

In [ ]:
# Initialize models and connections
print("Loading embedding model...")
model = SentenceTransformer(EMBEDDING_MODEL)

print("Loading FAISS index...")
vector_store = FaissStore(384)

print("Connecting to Neo4j...")
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

print(f"✓ FAISS index contains {vector_store.index.ntotal} vectors")

## 1. Vector Search with FAISS

In [ ]:
def vector_search(query, top_k=5):
    """Search for similar chunks using FAISS."""
    qvec = model.encode([query])[0]
    results = vector_store.search(qvec, k=top_k)
    
    data = []
    for idx, score, meta in results:
        data.append({
            'Chunk ID': meta.get('chunk_id'),
            'Score': f"{score:.4f}",
            'Text Preview': meta.get('text', '')[:150] + '...'
        })
    
    return pd.DataFrame(data)

In [ ]:
# Example: Search for environmental responsibility
query = "trách nhiệm bảo vệ môi trường"
print(f"Query: {query}\n")
vector_search(query, top_k=3)

## 2. Knowledge Graph Queries

In [ ]:
def run_cypher(query, params=None):
    """Execute a Cypher query and return results as DataFrame."""
    with driver.session() as session:
        result = session.run(query, params or {})
        return pd.DataFrame([dict(record) for record in result])

In [ ]:
# Count entities by type
query = """
MATCH (e:Entity)
RETURN e.type AS Type, count(*) AS Count
ORDER BY Count DESC
"""
run_cypher(query)

In [ ]:
# Find all articles
query = """
MATCH (e:Entity {type: 'Article'})
RETURN e.name AS Article, e.source_chunk AS Chunk
ORDER BY e.name
"""
run_cypher(query)

In [ ]:
# Find entity relationships
query = """
MATCH (a:Entity)-[r:REL]->(b:Entity)
RETURN a.name AS Entity1, a.type AS Type1, 
       r.type AS Relation,
       b.name AS Entity2, b.type AS Type2
LIMIT 20
"""
run_cypher(query)

## 3. Entity Context Retrieval

In [ ]:
def get_entity_context(entity_name):
    """Get chunks and related entities for a given entity."""
    query = """
    MATCH (e:Entity)-[:MENTIONED_IN]->(c:Chunk)
    WHERE e.name CONTAINS $name
    OPTIONAL MATCH (e)-[r:REL]-(related:Entity)
    RETURN e.name AS entity, e.type AS type,
           collect(DISTINCT c.text)[0..2] AS chunks,
           collect(DISTINCT related.name) AS related_entities
    """
    
    with driver.session() as session:
        result = session.run(query, name=entity_name)
        records = [dict(r) for r in result]
        
        if not records:
            print(f"No entity found matching: {entity_name}")
            return
        
        for rec in records:
            print(f"\n{'='*60}")
            print(f"Entity: {rec['entity']} ({rec['type']})")
            print(f"\nChunks mentioning this entity:")
            for i, chunk in enumerate(rec['chunks'], 1):
                print(f"  {i}. {chunk[:200]}...")
            print(f"\nRelated entities: {', '.join(rec['related_entities'][:5])}")

In [ ]:
# Example: Get context for Article 5
get_entity_context("Điều 5")

## 4. Hybrid Search (KG + Vector)

In [ ]:
def hybrid_search(query_text, top_k=5):
    """Combine vector search and graph search."""
    results = {}
    
    # Vector search
    print("Searching FAISS...")
    qvec = model.encode([query_text])[0]
    vec_results = vector_store.search(qvec, k=top_k)
    
    for idx, score, meta in vec_results:
        chunk_id = meta.get('chunk_id')
        results[chunk_id] = {
            'chunk_id': chunk_id,
            'text': meta.get('text', ''),
            'vector_score': score,
            'graph_score': 0,
            'entities': []
        }
    
    # Graph search
    print("Searching Knowledge Graph...")
    cypher = """
    MATCH (e:Entity)-[:MENTIONED_IN]->(c:Chunk)
    WHERE toLower(e.name) CONTAINS toLower($query)
    RETURN c.chunk_id AS chunk_id, c.text AS text,
           collect(DISTINCT {name: e.name, type: e.type}) AS entities
    LIMIT $limit
    """
    
    with driver.session() as session:
        graph_results = session.run(cypher, query=query_text, limit=top_k)
        
        for record in graph_results:
            chunk_id = record['chunk_id']
            if chunk_id in results:
                results[chunk_id]['graph_score'] = 1.0
                results[chunk_id]['entities'] = record['entities']
            else:
                results[chunk_id] = {
                    'chunk_id': chunk_id,
                    'text': record['text'],
                    'vector_score': 0,
                    'graph_score': 1.0,
                    'entities': record['entities']
                }
    
    # Combine scores (simple addition)
    for r in results.values():
        r['combined_score'] = r['vector_score'] + r['graph_score']
    
    # Sort by combined score
    sorted_results = sorted(results.values(), key=lambda x: x['combined_score'], reverse=True)
    
    return sorted_results[:top_k]

In [ ]:
# Example: Hybrid search
query = "môi trường"
results = hybrid_search(query, top_k=3)

for i, r in enumerate(results, 1):
    print(f"\n{'='*60}")
    print(f"Result {i}: {r['chunk_id']}")
    print(f"Vector Score: {r['vector_score']:.4f}")
    print(f"Graph Score: {r['graph_score']:.4f}")
    print(f"Combined: {r['combined_score']:.4f}")
    print(f"\nEntities: {[e['name'] for e in r['entities'][:3]]}")
    print(f"\nText: {r['text'][:200]}...")

## 5. Graph Statistics & Analysis

In [ ]:
# Most connected entities (degree centrality)
query = """
MATCH (e:Entity)-[r:REL]-()
RETURN e.name AS Entity, e.type AS Type, count(r) AS Connections
ORDER BY Connections DESC
LIMIT 10
"""
print("Most connected entities:")
run_cypher(query)

In [ ]:
# Entities mentioned in most chunks
query = """
MATCH (e:Entity)-[:MENTIONED_IN]->(c:Chunk)
RETURN e.name AS Entity, e.type AS Type, count(DISTINCT c) AS ChunkCount
ORDER BY ChunkCount DESC
LIMIT 10
"""
print("Entities mentioned most frequently:")
run_cypher(query)

## 6. Export & Analysis

In [ ]:
# Load entities from JSON
with open('outputs/entities.json', 'r', encoding='utf-8') as f:
    entities = json.load(f)

# Convert to DataFrame for analysis
entity_df = pd.DataFrame([
    {'id': k, **v} for k, v in entities.items()
])

print(f"Total entities: {len(entity_df)}")
print("\nEntities by type:")
entity_df['type'].value_counts()

In [ ]:
# Sample entities
entity_df.head(10)

## Cleanup

In [ ]:
# Close Neo4j connection
driver.close()
print("✓ Connections closed")